# Preprocess 2

### Plik ten skupia się na wyekstrachowaniu cech fali dźwiękowej bezpośrednio z utworów. 
Uzywamy tu całości utworu!
Output: plik extracted.csv, zawierający cechy oraz track_id

In [ ]:
import numpy as np
import pandas as pd
import librosa
from pathlib import Path
from pydub import AudioSegment
from typing import Dict, List, Optional

def _audiosegment_to_float32_mono(seg: AudioSegment, sr: int, clip_seconds: float | None) -> np.ndarray:
    seg = seg.set_channels(1).set_frame_rate(sr)
    if clip_seconds is not None:
        seg = seg[: int(clip_seconds * 1000)]  
    samples = np.array(seg.get_array_of_samples())
    max_val = float(2 ** (8 * seg.sample_width - 1))
    y = samples.astype(np.float32) / max_val
    return np.clip(y, -1.0, 1.0)

def _summ_stats(prefix: str, x: np.ndarray) -> Dict[str, float]:
    x = np.asarray(x)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return {f"{prefix}_mean": np.nan, f"{prefix}_std": np.nan}
    return {f"{prefix}_mean": float(np.mean(x)), f"{prefix}_std": float(np.std(x))}

def extract_fma_features_to_csv(
    root_dir: str | Path,
    out_csv: str | Path = "extracted.csv",
    sr: int = 16000,              
    n_mfcc: int = 13,
    clip_seconds: float | None = 30, 
    compute_tempo: bool = False,    
    max_files: Optional[int] = None,
) -> pd.DataFrame:
    root_dir = Path(root_dir)
    mp3_files = sorted(root_dir.rglob("*.mp3"))
    if max_files is not None:
        mp3_files = mp3_files[:max_files]

    rows: List[Dict[str, float | str]] = []

    for i, path in enumerate(mp3_files, 1):
        track_id = path.stem
        try:
            seg = AudioSegment.from_file(path, format="mp3")
            y = _audiosegment_to_float32_mono(seg, sr=sr, clip_seconds=clip_seconds)

            if y.size < sr: 
                continue

            duration_s = y.size / sr
            rms = librosa.feature.rms(y=y)[0]
            zcr = librosa.feature.zero_crossing_rate(y)[0]

            centroid = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
            bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
            rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr, roll_percent=0.85)[0]
            flatness = librosa.feature.spectral_flatness(y=y)[0]

            mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
            chroma = librosa.feature.chroma_stft(y=y, sr=sr)

            feats: Dict[str, float | str] = {"track_id": track_id, "duration_s": float(duration_s)} # ID oraz czas trwania
            feats.update(_summ_stats("rms", rms)) # średnia głośność / energia
            feats.update(_summ_stats("zcr", zcr)) # zróznicowanie
            feats.update(_summ_stats("centroid", centroid)) # środek masy częstotliwości 
            feats.update(_summ_stats("bandwidth", bandwidth)) # jak rozproszona jest częstotliwość wokół centroidu
            feats.update(_summ_stats("rolloff", rolloff))
            feats.update(_summ_stats("flatness", flatness)) # jak bardzo muza przypomina szum

            for k in range(n_mfcc):
                feats[f"mfcc{k+1}_mean"] = float(np.mean(mfcc[k]))
                feats[f"mfcc{k+1}_std"]  = float(np.std(mfcc[k]))

            for k in range(chroma.shape[0]):
                feats[f"chroma{k+1}_mean"] = float(np.mean(chroma[k]))
                feats[f"chroma{k+1}_std"]  = float(np.std(chroma[k]))

            if compute_tempo:
                onset_env = librosa.onset.onset_strength(y=y, sr=sr)
                tempo, _ = librosa.beat.beat_track(onset_envelope=onset_env, sr=sr)
                feats["tempo_bpm"] = float(tempo)

            rows.append(feats)

            if i % 200 == 0:
                print(f"{i}/{len(mp3_files)} processed")

        except Exception:
            continue

    df = pd.DataFrame(rows)
    if not df.empty:
        df = df[["track_id"] + [c for c in df.columns if c != "track_id"]]
    df.to_csv(out_csv, index=False, float_format="%.4f")
    return df


In [13]:
extract_fma_features_to_csv(root_dir = '../fma_tracks/fma_small')

200/8000 processed
400/8000 processed
600/8000 processed
800/8000 processed


/Users/pawelmozaryn/MACHINE_LEARNING/Nonlinear_and_ml/.venv/lib/python3.11/site-packages/librosa/core/pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(


1000/8000 processed
1200/8000 processed
1400/8000 processed


/Users/pawelmozaryn/MACHINE_LEARNING/Nonlinear_and_ml/.venv/lib/python3.11/site-packages/librosa/core/pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(
/Users/pawelmozaryn/MACHINE_LEARNING/Nonlinear_and_ml/.venv/lib/python3.11/site-packages/librosa/core/pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(
/Users/pawelmozaryn/MACHINE_LEARNING/Nonlinear_and_ml/.venv/lib/python3.11/site-packages/librosa/core/pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(
/Users/pawelmozaryn/MACHINE_LEARNING/Nonlinear_and_ml/.venv/lib/python3.11/site-packages/librosa/core/pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(
/Users/pawelmozaryn/MACHINE_LEARNING/Nonlinear_and_ml/.venv/lib/python3.11/site-packages/librosa/core/pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  r

1600/8000 processed


...

/Users/pawelmozaryn/MACHINE_LEARNING/Nonlinear_and_ml/.venv/lib/python3.11/site-packages/librosa/core/pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(


6200/8000 processed
6400/8000 processed
6600/8000 processed
6800/8000 processed
7000/8000 processed
7200/8000 processed


/Users/pawelmozaryn/MACHINE_LEARNING/Nonlinear_and_ml/.venv/lib/python3.11/site-packages/librosa/core/pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(


7400/8000 processed
7600/8000 processed
7800/8000 processed
8000/8000 processed


,track_id,duration_s,rms_mean,rms_std,zcr_mean,zcr_std,centroid_mean,centroid_std,bandwidth_mean,bandwidth_std,...,chroma8_mean,chroma8_std,chroma9_mean,chroma9_std,chroma10_mean,chroma10_std,chroma11_mean,chroma11_std,chroma12_mean,chroma12_std
0,000002,29.977,0.147649,0.073086,0.197228,0.094176,2576.968656,732.410426,2215.362306,247.724420,...,0.440590,0.280154,0.382463,0.267655,0.373707,0.276561,0.408052,0.268442,0.551780,0.278287
1,000005,30.000,0.151194,0.076020,0.121477,0.071330,2069.149088,664.415788,2184.762291,303.443802,...,0.410314,0.292314,0.474761,0.309509,0.496169,0.321605,0.491084,0.315126,0.432661,0.309723
2,000010,29.977,0.188524,0.040164,0.195863,0.033248,2176.973534,279.412776,1759.175343,188.451563,...,0.334926,0.171423,0.381010,0.273364,0.301069,0.210512,0.566932,0.280324,0.309867,0.196944
3,000140,29.977,0.069680,0.037031,0.053382,0.055046,1467.471858,793.223026,1969.234708,411.944054,...,0.208859,0.306380,0.165537,0.208020,0.259537,0.305940,0.246933,0.327065,0.183195,0.214552
4,000141,29.977,0.101561,0.077593,0.079607,0.057890,1489.010160,634.136635,1647.011219,371.161540,...,0.216088,0.312088,0.200390,0.236666,0.372674,0.340856,0.489017,0.415050,0.182940,0.198977
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7991,154308,29.977,0.069392,0.059978,0.091686,0.050792,1580.813023,635.470128,1647.774582,416.401235,...,0.294057,0.193496,0.711064,0.320022,0.340665,0.281150,0.295293,0.264288,0.275827,0.297171
7992,154309,29.977,0.076533,0.063449,0.130843,0.080725,2183.142621,1153.401351,1847.595918,696.466256,...,0.420745,0.356960,0.244341,0.292488,0.232085,0.331505,0.216072,0.280672,0.371458,0.383800
7993,154413,30.000,0.159018,0.026130,0.049727,0.037342,1191.593292,678.226670,1652.627589,576.839103,...,0.315749,0.280548,0.188448,0.163382,0.298778,0.297020,0.215360,0.137389,0.440363,0.328048
7994,154414,29.977,0.127599,0.032147,0.112288,0.058802,1929.797073,736.366109,2042.683604,370.967369,...,0.254369,0.290713,0.270887,0.272965,0.298302,0.320839,0.201945,0.226385,0.297031,0.333811
